<a href="https://colab.research.google.com/github/EV-MAX-INC/EV-FORKIN/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Pro and Pro+ subscriber Colab environments.  This means subscribers can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [1]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.0-flash',
 'google/gemini-2.0-flash-lite',
 'google/gemini-2.5-flash',
 'google/gemini-2.5-flash-lite',
 'google/gemini-2.5-pro',
 'google/gemma-3-12b',
 'google/gemma-3-1b',
 'google/gemma-3-27b',
 'google/gemma-3-4b']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [2]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.


In [3]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-2.0-flash-lite')
print(response)

The capital of England is London.



For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [4]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

Elias, a man woven from quiet routines and the scent of old paper, ran a dusty little bookstore nestled between a bustling cafe and a perpetually closed antique shop. His life was predictable, comforting in its sameness.

One Tuesday, while re-shelving a forgotten copy of "Moby Dick" that smelled vaguely of cinnamon and rain, his fingers brushed against something hard and cold tucked deep within the spine. He pulled it out.

It was a key.

Not an ordinary key, but one made of tarnished silver, intricately filigreed with what looked like tiny, unfurling fern fronds. It felt impossibly old, cool against his palm, and too large for any lock he knew. It had no numbers, no distinguishing marks, just the exquisite, silent promise of a secret.

Elias, usually stoic, felt a prickle of excitement. He tried it on his own front door – too big. His desk drawer – too intricate. The old cash register – completely wrong. He even tried the padlocks on the perpetually closed antique shop next door, muc

In [5]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    # OPTIMIZATION: Move punctuation set to class level to avoid recreating it in every call
    # This reduces memory allocations from O(n_words) to O(1)
    NO_LEADING_SPACE_PUNCTUATION = frozenset({
        ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
        ")", "]", "}",                     # Closing brackets
        "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
        "'m", "'M", "'ll", "'LL", "'d", "'D",
        "n't", "N'T",
        "...", "…"                          # Ellipses
    })

    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        # OPTIMIZATION: Use list to buffer output, then write once at the end
        # This reduces system calls and improves performance
        output_buffer = []
        i = 0
        n = len(text_chunk)
        
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                word_len = len(current_word)  # OPTIMIZATION: Cache length
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if word_len > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        output_buffer.append('\n')
                        self.current_line_length = 0
                    
                    # OPTIMIZATION: Break long word using slicing instead of char-by-char iteration
                    # This is more efficient for very long words
                    pos = 0
                    while pos < word_len:
                        if self.current_line_length >= self.max_length:
                            output_buffer.append('\n')
                            self.current_line_length = 0
                        
                        chunk_size = min(self.max_length - self.current_line_length, word_len - pos)
                        output_buffer.append(current_word[pos:pos + chunk_size])
                        self.current_line_length += chunk_size
                        pos += chunk_size
                        
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + word_len > self.max_length:
                    output_buffer.append('\n')
                    output_buffer.append(current_word)
                    self.current_line_length = word_len
                    
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # OPTIMIZATION: Use class-level frozenset for O(1) lookup
                        if current_word not in self.NO_LEADING_SPACE_PUNCTUATION:
                            output_buffer.append(' ')
                            self.current_line_length += 1
                    output_buffer.append(current_word)
                    self.current_line_length += word_len

            if delimiter == '\n':
                output_buffer.append('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    output_buffer.append('\n')
                    self.current_line_length = 0

        # OPTIMIZATION: Single write call instead of multiple sys.stdout.write calls
        # This significantly reduces system call overhead
        sys.stdout.write(''.join(output_buffer))
        sys.stdout.flush()


In [6]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)

Alright, buckle up, because the evolution of the Roman Empire is less of a
straight line and more of a tangled, glorious, bloody, and fascinating tapestry
woven across centuries. We' ll start from its humble beginnings, trace its
explosive growth, and watch its eventual fragmentation into distinct entities.

**From Village Republic to Mediterranean Powerhouse (7 53 BCE - 27 BCE):**

It all began, as legend has it, with Romulus and Remus, twin brothers suckled by
a she -wolf, in 753 BCE. From this small, agrarian settlement on the Palatine
Hill along the Tiber River, Rome grew slowly, absorbing surrounding villages and
consolidating power. This wasn't empire yet, mind you, but a *Republic *. Think
of it as a society obsessed with civic duty and obsessed with the idea of *not*
having a king. Power was ostensibly distributed among elected magistrates
(Consuls were the big shots, serving one-year terms), the Senate (an advisory
body composed of elder statesmen, wielding considerable influe